# Aspect-Based Sentiment Analysis sử dụng Softmax Regression

Trong notebook này, chúng ta tiếp cận bài toán Khai phá ý kiến (Aspect-Based Sentiment Analysis) cho bộ dữ liệu tiếng Việt (VLSP 2018 - Restaurant). Mô hình phân loại được yêu cầu sử dụng là **Softmax Regression** (tương đương Logistic Regression đa lớp).

Ta xây dựng mô hình:
- Mỗi nhãn phân loại (aspect) là tổ hợp giữa **Entity** (Thực thể) và **Attribute** (Thuộc tính). Ví dụ: `FOOD#QUALITY`.
- Cần phân loại 4 trạng thái cho mỗi aspect: `positive`, `negative`, `neutral` (trung tính), và `null` (không được nhắc đến trong câu/nhạc nhiên).
- Tập mô hình sẽ gồm nhiều mô hình nhỏ, số lượng bằng đúng số lượng aspect có thể có.

## Trích xuất đặc trưng (Feature Engineering)
Ta sử dụng **Mô hình Túi Từ (Bag of Words / CountVectorizer) & TF-IDF (Term Frequency - Inverse Document Frequency)**: Việc kết hợp TF-IDF với **n-gram** (thông dụng là 1-gram và 2-gram) giúp bảo lưu thứ tự từ căn bản (ví dụ bắt được các cụm từ ý nghĩa như "không ngon", "rất tốt") mà đơn giản, cực nhanh và vẫn đảm bảo hiệu suất rất tốt đối với Softmax Regression.

In [1]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import warnings

warnings.filterwarnings('ignore')

### 1. Đọc và tiền xử lý dữ liệu
Dựa vào thông tin cấu trúc các file trong dữ liệu VLSP 2018 (Train, Dev, Test):
- Dòng chứa index của bình luận.
- Dòng chứa nội dung đánh giá.
- Dòng chứa dán nhãn (ví dụ: `{FOOD#QUALITY, positive}`).
- Dòng trống phân chia block.

Chúng ta sẽ định nghĩa hàm read file để trích xuất text cùng các cặp aspect-polarity từ các file.

In [2]:
def load_data(filepath):
    texts = []
    labels = []
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.read().strip().split('\n')
        
    # Mỗi khối dữ liệu bình luận chiếm 4 dòng
    for i in range(0, len(lines), 4):
        if i + 2 >= len(lines):
            break
        text = lines[i+1].strip()
        label_line = lines[i+2].strip()
        
        aspect_dict = {}
        if label_line:
            # Sử dụng regular expression để parse nhanh các cụm {ASPECT, polarity}
            matches = re.findall(r'\{([^,]+),\s*([^}]+)\}', label_line)
            for aspect, polarity in matches:
                aspect_dict[aspect.strip()] = polarity.strip()
                
        texts.append(text)
        labels.append(aspect_dict)
        
    return texts, labels

# Khai báo đúng đường dẫn đến thư mục dữ liệu
train_texts, train_labels = load_data('VLSP2018-SA-train-dev-test/1-VLSP2018-SA-Restaurant-train (7-3-2018).txt')
dev_texts, dev_labels = load_data('VLSP2018-SA-train-dev-test/2-VLSP2018-SA-Restaurant-dev (7-3-2018).txt')
test_texts, test_labels = load_data('VLSP2018-SA-train-dev-test/3-VLSP2018-SA-Restaurant-test (8-3-2018).txt')

print(f"Số lượng đánh giá trong tập Train: {len(train_texts)}")
print(f"Số lượng đánh giá trong tập Dev: {len(dev_texts)}")
print(f"Số lượng đánh giá trong tập Test: {len(test_texts)}")


print(f"Ta in ra một ví dụ để coi thử đã trích xuất dữ liệu như mong muốn chưa:")
print(f"Câu ví dụ: {train_texts[0]}")
print(f"Nhãn tương ứng: {train_labels[0]}")

Số lượng đánh giá trong tập Train: 2961
Số lượng đánh giá trong tập Dev: 1290
Số lượng đánh giá trong tập Test: 500
Ta in ra một ví dụ để coi thử đã trích xuất dữ liệu như mong muốn chưa:
Câu ví dụ: _ Ảnh chụp từ hôm qua, đi chơi với gia đình và 1 nhà họ hàng đang sống tại Sài Gòn. _ Hôm qua đi ăn trưa muộn, ai cũng đói hết nên lúc có đồ ăn là nhào vô ăn liền, bởi vậy mới quên chụp các phần gọi thêm với nước mắm, chỉ chụp món chính thôi! _ Đói quá nên không biết đánh giá đồ ăn kiểu gì luôn 😅😅😅_ Chọn cái này vì thấy nó lạ với tui.
Nhãn tương ứng: {'FOOD#STYLE&OPTIONS': 'neutral', 'FOOD#QUALITY': 'neutral'}


### 2. Định nghĩa Aspect và Chuẩn bị nhãn lớp cho từng Base Model
Duyệt qua tập Train để tìm tất cả các **tổ hợp Entity#Attribute** có thể xuất hiện.
Mỗi một Aspect sẽ tương đương một mô hình mang nhãn y riêng. Với aspect không xuất hiện với mỗi dòng review, ta sẽ cho kết quả của aspect đó của review đó là `null`.

In [3]:
# Danh sách Entity và Attribute từ Hướng dẫn VLSP 2018 (Guidelines-SA-Restaurant)
entities = [
    "RESTAURANT", 
    "AMBIENCE", 
    "LOCATION", 
    "FOOD", 
    "SERVICE", 
    "DRINKS"
]

attributes = [
    "GENERAL", 
    "PRICES", 
    "QUALITY", 
    "STYLE&OPTIONS", 
    "MISCELLANEOUS"
]

# Tạo tất cả các tổ hợp có thể có
all_aspects = sorted([f"{e}#{a}" for e in entities for a in attributes])

print(f"Tổng số mô hình (Aspects) có thể xây dựng: {len(all_aspects)}")

def extract_aspect_labels(labels_list, aspects):
    """
    Hàm biến đổi danh sách Label tổng ra cấu trúc nhóm các Label theo mỗi Aspect.
    Đầu ra dạng: {'FOOD#QUALITY': ['positive', 'null', 'negative', ...], 'RESTAURANT...': [...],...}
    """
    y_dict = {aspect: [] for aspect in aspects}
    for label_dict in labels_list:
        for aspect in aspects:
            # Gán fallback chuỗi 'null' nếu đánh giá ko chứa aspect này (ko đề cập) 
            y_dict[aspect].append(label_dict.get(aspect, 'null'))
    return y_dict

y_train_dict = extract_aspect_labels(train_labels, all_aspects)
y_dev_dict = extract_aspect_labels(dev_labels, all_aspects)
y_test_dict = extract_aspect_labels(test_labels, all_aspects)

Tổng số mô hình (Aspects) có thể xây dựng: 30


### 3. Feature Extraction (Trích xuất đặc trưng với TF-IDF)
Tiến hành fitting TF-IDF trên tập Train, sau đó map (với transform) đặc trưng văn bản ra không gian ma trận thưa cho tập Dev và Test.

In [4]:
# Lấy TF-IDF cùng unigram và bi-gram, lọc các từ xuất hiện tối thiểu 2 lần
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100000)
X_train = vectorizer.fit_transform(train_texts)

X_dev = vectorizer.transform(dev_texts)
X_test = vectorizer.transform(test_texts)

print("Kích Thước bộ vector đặc trưng không gian (X_train):", X_train.shape)

Kích Thước bộ vector đặc trưng không gian (X_train): (2961, 21476)


### 4. Tìm kiếm Siêu tham số (Hyperparameter Tuning) trên tập Dev
Thực hiện Grid Search để tìm tham số `C` (hệ số điều chuẩn - regularization strength) tốt nhất cho từng mô hình Aspect dựa trên độ đo trên tập Dev.

In [5]:
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier

models = {}
best_params = {}

# Tập các giá trị C cần thử (Giá trị C càng nhỏ, mô hình càng bị regularized mạnh)
C_values = [0.01, 0.1, 1, 10, 100]

print("Đang tìm kiếm siêu tham số tốt nhất trên tập Dev...")

for aspect in all_aspects:
    y_train_aspect = y_train_dict[aspect]
    y_dev_aspect = y_dev_dict[aspect]
    
    unique_train_classes = set(y_train_aspect)
    
    # Nếu aspect này KHÔNG XUẤT HIỆN trong tập train (chỉ có duy nhất class 'null')
    # Ta sử dụng DummyClassifier để luôn dự đoán 'null' như một dạng default weight (đoán mò)
    if len(unique_train_classes) == 1:
        dummy_model = DummyClassifier(strategy='constant', constant=list(unique_train_classes)[0])
        dummy_model.fit(X_train, y_train_aspect)
        models[aspect] = dummy_model
        best_params[aspect] = "N/A (Dummy)"
        continue
    
    best_C = 1.0
    best_acc = -1.0
    
    for c_val in C_values:
        # Train model nháp với giá trị C hiện tại
        temp_model = LogisticRegression(C=c_val, solver='lbfgs', max_iter=300)
        temp_model.fit(X_train, y_train_aspect)
        
        # Predict trên tập dev để lấy accuracy
        preds = temp_model.predict(X_dev)
        acc = accuracy_score(y_dev_aspect, preds)
        
        if acc > best_acc:
            best_acc = acc
            best_C = c_val
            
    best_params[aspect] = best_C
    
    # Train lại model cuối cùng trên tập Train với best_C đã tìm được
    final_model = LogisticRegression(C=best_C, solver='lbfgs', max_iter=300)
    final_model.fit(X_train, y_train_aspect)
    models[aspect] = final_model
    
print("Hoàn tất việc tối ưu và huấn luyện trên hệ mô hình!")

print("\n--- CÁC HYPERPARAM (C) TỐT NHẤT TÌM ĐƯỢC CHO MỖI ASPECT ---")
for aspect, opt_c in best_params.items():
    print(f"{aspect:<30} | Best C: {opt_c}")

Đang tìm kiếm siêu tham số tốt nhất trên tập Dev...
Hoàn tất việc tối ưu và huấn luyện trên hệ mô hình!

--- CÁC HYPERPARAM (C) TỐT NHẤT TÌM ĐƯỢC CHO MỖI ASPECT ---
AMBIENCE#GENERAL               | Best C: 100
AMBIENCE#MISCELLANEOUS         | Best C: N/A (Dummy)
AMBIENCE#PRICES                | Best C: N/A (Dummy)
AMBIENCE#QUALITY               | Best C: N/A (Dummy)
AMBIENCE#STYLE&OPTIONS         | Best C: N/A (Dummy)
DRINKS#GENERAL                 | Best C: N/A (Dummy)
DRINKS#MISCELLANEOUS           | Best C: N/A (Dummy)
DRINKS#PRICES                  | Best C: 100
DRINKS#QUALITY                 | Best C: 10
DRINKS#STYLE&OPTIONS           | Best C: 0.01
FOOD#GENERAL                   | Best C: N/A (Dummy)
FOOD#MISCELLANEOUS             | Best C: N/A (Dummy)
FOOD#PRICES                    | Best C: 100
FOOD#QUALITY                   | Best C: 100
FOOD#STYLE&OPTIONS             | Best C: 100
LOCATION#GENERAL               | Best C: 100
LOCATION#MISCELLANEOUS         | Best C: N/A (Dummy

### 5. Dự đoán và Evaluate các Mô Hình
Chạy Prediction kiểm thử trên bộ tập Test cho từng model. Và áp dụng metric Classification Report (Precision, Recall, F1-Score) để đánh giá các phân lớp polarities.

In [6]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate_models(X, y_dict_true, aspects, models):
    print(f"{'='*60}")
    print(f"{'ASPECT':<30} | {'ACCURACY':<10} | {'MACRO F1':<10}")
    print(f"{'-'*60}")
    
    acc_list = []
    f1_list = []
    
    for aspect in aspects:
        model = models[aspect]
        y_true = y_dict_true[aspect]
        y_pred = model.predict(X)
        
        unique_labels = set(y_true)
        # Chỉ đánh giá các aspect có xuất hiện trong test set
        if len(unique_labels) > 1 or 'null' not in unique_labels:
            acc = accuracy_score(y_true, y_pred)
            f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
            
            acc_list.append(acc)
            f1_list.append(f1)
            
            print(f"{aspect:<30} | {acc:.4f}     | {f1:.4f}")
            
    print(f"{'='*60}")
    if len(acc_list) > 0:
        print(f"{'TRUNG BÌNH TỔNG THỂ':<30} | {sum(acc_list)/len(acc_list):.4f}     | {sum(f1_list)/len(f1_list):.4f}")

# Chạy qua các model test
evaluate_models(X_test, y_test_dict, all_aspects, models)


ASPECT                         | ACCURACY   | MACRO F1  
------------------------------------------------------------
AMBIENCE#GENERAL               | 0.7540     | 0.4254
DRINKS#PRICES                  | 0.8500     | 0.2797
DRINKS#QUALITY                 | 0.8760     | 0.3037
DRINKS#STYLE&OPTIONS           | 0.9080     | 0.2379
FOOD#PRICES                    | 0.5140     | 0.4369
FOOD#QUALITY                   | 0.8360     | 0.4235
FOOD#STYLE&OPTIONS             | 0.7140     | 0.3366
LOCATION#GENERAL               | 0.7060     | 0.4248
RESTAURANT#GENERAL             | 0.5560     | 0.1831
RESTAURANT#MISCELLANEOUS       | 0.7400     | 0.2126
RESTAURANT#PRICES              | 0.8540     | 0.2303
SERVICE#GENERAL                | 0.7700     | 0.4555
TRUNG BÌNH TỔNG THỂ            | 0.7565     | 0.3292
